In [38]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import shap

np.random.seed(42)

# 1. LOAD SYNTHETIC DATA

In [39]:
df = pd.read_csv("./data/synthetic_robo_data.csv") 

In [40]:
# TARGET
y = df["investor_type"]

# FEATURES
X = df.drop(columns=["investor_type"])

# 2. DEFINE NUMERIC + CATEGORICAL FEATURES
# (based on synthetic dataset structure)

In [41]:
numeric = [
    "age",
    "annual_income",
    "net_worth",
    "savings_rate",
    "debt_ratio",
    "employment_years",
    "investment_experience_years",
    "time_horizon_years",
    "emergency_fund_months",
    "risk_tolerance",
    "risk_capacity",
    "combined_score",
    "curr_stock",
    "curr_bonds",
    "curr_cash",
    "curr_alts"
]

categorical = [
    "employment_status",
    "primary_goal",
    "liquidity_need",
    "resources_used",
]

# 3. PREPROCESSING PIPELINE

In [42]:
preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
        ("num", StandardScaler(), numeric),
    ]
)

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    random_state=42
)

clf = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", model)
])

# 4. TRAIN / TEST SPLIT + TRAIN

In [43]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

clf.fit(X_train, y_train)

print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(y_test, clf.predict(X_test)))


=== CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

  aggressive       1.00      1.00      1.00        50
conservative       1.00      1.00      1.00        11
    moderate       1.00      1.00      1.00      1189

    accuracy                           1.00      1250
   macro avg       1.00      1.00      1.00      1250
weighted avg       1.00      1.00      1.00      1250



# 5. GENETIC ALGORITHM PORTFOLIO OPTIMIZER

In [44]:
def ga_optimize(mu, Sigma, risk_aversion, 
                pop_size=100, generations=200, 
                retain=0.3, random_select=0.1, mutate_chance=0.1):

    mu = np.array(mu).flatten()
    Sigma = np.array(Sigma)

    def utility(weights):
        weights = np.array(weights)
        r = float(np.dot(weights, mu))
        risk = float(np.dot(weights.T, np.dot(Sigma, weights)))
        return float(r - risk_aversion * risk)

    def create_individual():
        w = np.random.rand(len(mu))
        w /= w.sum()
        return w

    def mutate(ind):
        idx = np.random.randint(len(ind))
        ind[idx] += np.random.normal(0, 0.1)
        ind[ind < 0] = 0
        ind /= ind.sum()
        return ind

    population = [create_individual() for _ in range(pop_size)]

    for _ in range(generations):

        graded = [(utility(ind), ind) for ind in population]
        graded.sort(key=lambda x: x[0], reverse=True)

        retain_len = int(retain * pop_size)
        parents = [ind for _, ind in graded[:retain_len]]

        # random survivor selection
        for _, ind in graded[retain_len:]:
            if np.random.rand() < random_select:
                parents.append(ind)

        children = []
        needed = pop_size - len(parents)

        while len(children) < needed:
            p1 = parents[np.random.randint(len(parents))]
            p2 = parents[np.random.randint(len(parents))]
            cut = np.random.randint(1, len(mu))
            child = np.concatenate([p1[:cut], p2[cut:]])
            child /= child.sum()
            children.append(child)

        population = parents + [
            mutate(child) if np.random.rand() < mutate_chance else child
            for child in children
        ]

    best = max(population, key=lambda ind: utility(ind))

    return {
        "weights": best,
        "score": utility(best)
    }



# 6. PORTFOLIO PARAMETERS

In [45]:
assets = ["Stocks", "Mutual Funds", "Crypto", "Cash", "Bonds"]
mu = np.array([0.13, 0.09, 0.20, 0.03, 0.055])
vol = np.array([0.22, 0.12, 0.40, 0.05, 0.02])
corr = np.eye(5)
Sigma = np.outer(vol, vol) * corr

risk_map = {
    "conservative": 22.0,
    "moderate": 8.0,
    "aggressive": 2.0
}

# 7. NEW USER EXAMPLE (synthetic structure)

In [46]:
new_user = {
    "age": 29,
    "annual_income": 45000,
    "net_worth": 15000,
    "savings_rate": 0.18,
    "debt_ratio": 0.22,
    "employment_years": 3,
    "investment_experience_years": 1,
    "time_horizon_years": 20,
    "emergency_fund_months": 4,

    "employment_status": "employee",
    "primary_goal": "wealth_generation",
    "liquidity_need": "medium",
    "resources_used": "online",

    # Psychometrics
    "risk_tolerance": 0.95,
    "risk_capacity": 0.40,
    "combined_score": 0.49,

    # Current allocation (optional)
    "curr_stock": 0.30,
    "curr_bonds": 0.40,
    "curr_cash": 0.20,
    "curr_alts": 0.10,
}

new_df = pd.DataFrame([new_user])

pred_class = clf.predict(new_df)[0]
print("\nPredicted Investor Type:", pred_class)

risk_aversion = risk_map[pred_class]
portfolio = ga_optimize(mu, Sigma, risk_aversion)

print("\n--- OPTIMIZED PORTFOLIO ---")
for asset, weight in zip(assets, portfolio["weights"]):
    print(f"{asset}: {weight:.3f}")


Predicted Investor Type: moderate

--- OPTIMIZED PORTFOLIO ---
Stocks: 0.102
Mutual Funds: 0.171
Crypto: 0.058
Cash: 0.000
Bonds: 0.669


# 8. SHAP EXPLANABILITY

In [47]:
# 1. Extract the preprocessed training data
# We use the 'preprocess' step of the trained pipeline to transform the training data.
X_train_transformed = clf["preprocess"].transform(X_train)

# 2. Get feature names for the transformed data
# A list of all one-hot encoded and scaled feature names
cat_features = clf["preprocess"].named_transformers_["cat"].get_feature_names_out(categorical).tolist()
all_features = cat_features + numeric
X_train_transformed_df = pd.DataFrame(X_train_transformed, columns=all_features)

# 3. Extract the trained Random Forest model
model_rf = clf["model"]

# 4. Create the Tree Explainer
# Using the preprocessed training data as the background dataset
explainer = shap.TreeExplainer(model_rf, X_train_transformed_df)

In [48]:
# 1. Preprocess the new user's data
new_user_transformed = clf["preprocess"].transform(new_df)
new_user_transformed_df = pd.DataFrame(new_user_transformed, columns=all_features)

# 2. Calculate SHAP values for the new user
# SHAP values are calculated for each class (conservative, moderate, aggressive).
shap_values = explainer.shap_values(new_user_transformed_df)

In [49]:
# Check the actual classes the model knows about
model_classes = clf["model"].classes_
print(f"Model Classes: {model_classes}")
print(f"Number of Classes: {len(model_classes)}")

Model Classes: ['aggressive' 'conservative' 'moderate']
Number of Classes: 3


In [50]:
# Check the SHAP output structure
print(f"Type of SHAP values: {type(shap_values)}")
# If it's a list (multi-class), print the shape of the first element
if isinstance(shap_values, list):
    print(f"SHAP values list size (number of classes): {len(shap_values)}")
    if len(shap_values) > 0:
        print(f"Shape of first class array: {shap_values[0].shape}")
# If it's a single numpy array (binary/error), print its shape
else:
    print(f"SHAP values shape (binary or error): {shap_values.shape}")

Type of SHAP values: <class 'numpy.ndarray'>
SHAP values shape (binary or error): (1, 34, 3)


In [53]:
# Assuming the variables from your previous steps are still active:
# pred_class: 'moderate'
# class_names: ['aggressive' 'conservative' 'moderate']
# shap_values: numpy array of shape (1, 34, 3)

# 1. Find the index of the predicted class ('moderate')
pred_class = clf.predict(new_df)[0]
class_names = clf.classes_
pred_class_idx = np.where(class_names == pred_class)[0][0] # Should be 2 for 'moderate'

# 2. Extract the SHAP values for the predicted class
# Indexing: [0] removes the sample dimension (size 1)
#          [:, pred_class_idx] selects the values for the correct class (index 2)
# The result will be a (34,) shape array of feature contributions.
shap_values_pred_class = shap_values[0][:, pred_class_idx]

# 3. Create a Series for easy viewing and sorting
shap_series = pd.Series(
    shap_values_pred_class, 
    index=all_features
).sort_values(key=abs, ascending=False)

print(f"\n--- TOP 10 SHAP CONTRIBUTORS FOR '{pred_class.upper()}' PREDICTION (Fixed) ---")
print(shap_series.head(30))

# The rest of your code to display the final portfolio weights remains the same


--- TOP 10 SHAP CONTRIBUTORS FOR 'MODERATE' PREDICTION (Fixed) ---
curr_stock                        -0.276688
curr_bonds                        -0.126614
curr_cash                         -0.022324
combined_score                     0.016774
net_worth                         -0.006318
liquidity_need_medium             -0.003448
annual_income                     -0.003333
savings_rate                      -0.002904
primary_goal_wealth_generation    -0.002839
age                               -0.002483
employment_years                  -0.001974
liquidity_need_low                 0.001971
investment_experience_years        0.001602
debt_ratio                        -0.001083
primary_goal_retirement            0.000924
time_horizon_years                -0.000708
resources_used_books              -0.000552
resources_used_online              0.000537
emergency_fund_months             -0.000519
risk_tolerance                     0.000512
liquidity_need_high                0.000396
curr_alt